# CUB-200-2011 birds — test-set CAM audit

This notebook is the visual companion to the [full benchmark](../results/cub/RESULTS.md).
It shows **three frozen test examples per architecture** using the exact same
center-cropped input for CE, FCAL V1, FCAL V2, and—where available—Universal FCAL.

How to read each panel:

- green/red text means the predicted class is correct/incorrect;
- the white contour is the annotated foreground;
- `inside` is the share of positive **predicted-class CAM** mass inside that contour;
- CAM colors are normalized within each panel and use `jet` (blue = low,
  red = high).

The examples are selected reproducibly from saved test arrays: first a strong
localization comparison where CE and V2 are both correct, then a CE→V2
classification rescue when one exists, then another localization improvement.
This is a qualitative audit, not an accuracy estimate; use the full result tables
for quantitative claims.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ASSETS = ROOT / "assets" / "result_showcases" / "cub"
manifest = pd.read_csv(ASSETS / "examples.csv")
print(f"Loaded {len(manifest)} frozen examples from {manifest['architecture'].nunique()} architectures")
manifest[["family", "architecture", "case_rank", "image_id", "true_class"]]

Loaded 18 frozen examples from 6 architectures


,family,architecture,case_rank,image_id,true_class
0,CNN,ResNet-50,1,5082,Mallard
1,CNN,ResNet-50,2,7470,Seaside Sparrow
2,CNN,ResNet-50,3,11600,Marsh Wren
3,CNN,DenseNet-121,1,6722,Brewer Sparrow
4,CNN,DenseNet-121,2,5873,White Pelican
5,CNN,DenseNet-121,3,7498,Seaside Sparrow
6,CNN,EfficientNet-B0,1,1901,Gray crowned Rosy Finch
7,CNN,EfficientNet-B0,2,10506,Tennessee Warbler
8,CNN,EfficientNet-B0,3,7344,Nelson Sharp tailed Sparrow
9,ViT,ViT-Tiny/16,1,10532,Tennessee Warbler


## CNNs

These CAMs are exact final-feature class activation maps: the classifier's own
linear weights projected back onto the final convolutional feature grid.

### ResNet-50

![ResNet-50 test cases](../assets/result_showcases/cub/resnet50_test_cases.png)

### DenseNet-121

![DenseNet-121 test cases](../assets/result_showcases/cub/densenet121_test_cases.png)

### EfficientNet-B0

![EfficientNet-B0 test cases](../assets/result_showcases/cub/efficientnet-b0_test_cases.png)

## Vision Transformers

The ViT runs use average-pooled patch tokens, so their class-specific patch CAMs
decompose the actual classifier score rather than substituting a generic
self-attention visualization.

### ViT-Tiny/16

![ViT-Tiny/16 test cases](../assets/result_showcases/cub/vit-tiny-p16-avg_test_cases.png)

### ViT-Small/16

![ViT-Small/16 test cases](../assets/result_showcases/cub/vit-small-p16-avg_test_cases.png)

### ViT-Base/16

![ViT-Base/16 test cases](../assets/result_showcases/cub/vit-base-p16-avg_test_cases.png)

## Selected-example audit table

The table below exposes every displayed prediction and inside-foreground value.
Because these examples were deliberately selected for visual diagnosis, do not
average this table or present it as benchmark performance.

In [2]:
display_columns = [
    column for column in manifest.columns
    if column in {"architecture", "case_rank", "image_id", "true_class"}
    or column.endswith("_correct")
    or column.endswith("_inside_fraction")
]
manifest[display_columns].style.format({
    column: "{:.1%}" for column in display_columns if column.endswith("_inside_fraction")
})

,architecture,case_rank,image_id,true_class,ce_correct,ce_inside_fraction,fcal_v1_correct,fcal_v1_inside_fraction,fcal_v2_correct,fcal_v2_inside_fraction
0,ResNet-50,1,5082,Mallard,True,43.3%,False,93.4%,True,87.4%
1,ResNet-50,2,7470,Seaside Sparrow,False,90.4%,False,93.3%,True,88.8%
2,ResNet-50,3,11600,Marsh Wren,True,13.3%,True,31.5%,True,79.6%
3,DenseNet-121,1,6722,Brewer Sparrow,True,28.2%,True,33.9%,True,94.7%
4,DenseNet-121,2,5873,White Pelican,False,81.7%,True,58.1%,True,92.8%
5,DenseNet-121,3,7498,Seaside Sparrow,True,11.7%,True,4.6%,True,55.9%
6,EfficientNet-B0,1,1901,Gray crowned Rosy Finch,True,27.7%,True,38.2%,True,68.0%
7,EfficientNet-B0,2,10506,Tennessee Warbler,False,10.7%,False,23.8%,True,55.5%
8,EfficientNet-B0,3,7344,Nelson Sharp tailed Sparrow,True,5.2%,True,32.2%,True,47.8%
9,ViT-Tiny/16,1,10532,Tennessee Warbler,True,30.8%,True,31.4%,True,36.6%


## Artifact provenance

The committed PNGs were reconstructed from the real best-validation checkpoints.
The accompanying `examples.csv` manifest freezes every displayed test ID,
prediction, correctness flag, confidence, and inside-foreground CAM value, so
this notebook remains auditable without downloading checkpoints or datasets.